In [1]:
!pip install ultralytics opencv-python deep-sort-realtime

In [3]:
# Import Libraries
import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO # YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort # DeepSORT
import time

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as keras_preprocess
from tensorflow.keras.preprocessing.image import img_to_array

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [4]:
def preprocess_frame(frame, resize_dim=(640, 480), denoise=False):
    resized_frame = cv2.resize(frame, resize_dim)  # Resize frame for consistent processing
    if denoise:
        resized_frame = cv2.fastNlMeansDenoisingColored(resized_frame, None, 10, 10, 7, 21)  # Optional noise reduction
    return resized_frame  # Return processed frame

In [5]:
def filter_vehicle_detections(yolo_result_boxes, class_names, confidence_threshold=0.3):
    detections = []
    for box in yolo_result_boxes:
        cls_id = int(box.cls[0])  # YOLO class ID
        conf = float(box.conf[0])  # YOLO confidence
        if conf < confidence_threshold:
            continue  # Skip low-confidence detections
        if cls_id in class_names:
            x1, y1, x2, y2 = box.xyxy[0]  # Bounding box coordinates
            detections.append(([int(x1), int(y1), int(x2), int(y2), conf], cls_id))  # Append detection if vehicle
    return detections  # Return filtered detections

In [6]:
def point_in_roi(point, roi_polygon):
    """Return True if the point (x,y) is inside the polygon (as np.array of (N,2))."""
    return cv2.pointPolygonTest(roi_polygon, (int(point[0]), int(point[1])), False) >= 0  # Use OpenCV for point-in-polygon test

In [1]:
def main(video_source=0,
         resize_dim=(640, 480),
         denoise=False,
         roi_points=None,
         output_video="annotated_output.avi",
         detection_interval=1,
         confidence_threshold=0.4):
    # --- COCO IDs for vehicles ---
    class_names = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}  # Valid YOLO vehicle classes
    custom_classes = [
        "Commercial Vehicles",
        "High-End Vehicles",
        "Low-End Vehicles",
        "Mid-Range Vehicles",
        "Motorcycle"
    ]
    MIN_W, MIN_H = 1, 1  # Minimum crop size for classification
    THRESHOLD = 0.5        # Min confidence for classification

    # ---- Model loading ----
    model = YOLO("yolo11n.pt")  # Load YOLO model 
    classifier_model = tf.keras.models.load_model("mobilenetv3_original.keras")  # Load trained MobileNetV3

    # ---- Video and Writer ----
    cap = cv2.VideoCapture(video_source)  # Open video or camera
    if not cap.isOpened():
        print(f"Error: Cannot open video source {video_source}")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0  # Get FPS or default to 30
    # out_width, out_height = resize_dim  # Output video dimensions
    # fourcc = cv2.VideoWriter_fourcc(*"XVID")
    # out_writer = cv2.VideoWriter(output_video, fourcc, fps, (out_width, out_height))  # Writer for annotated output video

    # ---- DeepSORT ----
    deepsort = DeepSort(
        max_age=15,              # Max number of missed detections before a track is deleted
        n_init=2,                # Minimum detections before a track is confirmed
        nms_max_overlap=1.0,    
        max_cosine_distance=0.2, # Controls appearance matching
        nn_budget=25,            # Embedding queue size
        embedder="mobilenet",    # Embedding network for tracking
        embedder_gpu=True        # Use GPU for embeddings if available
    )

    # ---- ROI polygon setup ----
    if roi_points is None:
        # Default: middle half of frame as ROI for counting
        w, h = resize_dim
        y0 = int(h/4)      # e.g. 480/4= 120
        y1 = int(h*3/4)   # e.g. 480*3/4= 360
        x0 = int(w/4)     # e.g. 640/4= 160
        x1 = int(w*3/4)   # e.g. 640*3/4= 480
        roi_points = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]  # Rectangle
        
    roi_polygon = np.array(roi_points, dtype=np.int32)    # OpenCV polygon

    # ---- State tracking ----
    track_classes = {}   # track_id -> predicted vehicle class
    track_memory = {}    # track_id -> {"inside_roi": False, "counted": False}
    class_counts = {name: 0 for name in custom_classes}  # Total counts per vehicle class
    class_counts["Unclassified"] = 0
    frame_idx = 0
    paused = False
    start_time = time.time()

    print(f"[INFO] ROI Polygon: {roi_points}")  # Log the ROI in use

    while True:
        if not paused:
            ret, frame = cap.read()  # Read next frame
            if not ret:
                print("End of video stream or cannot fetch the frame.")
                break
            if frame is None or frame.size == 0:
                print("Empty frame, skipping...")
                continue
            frame_idx += 1

            # -- Preprocess --
            preprocessed_frame = preprocess_frame(frame, resize_dim, denoise)  # Resize and denoise

            # -- Detection & Tracking --
            if frame_idx % detection_interval == 0:  # Only run YOLO every N frames
                yolo_results = model(preprocessed_frame, verbose=False)
                detections_for_tracker = []
                for result in yolo_results:
                    filtered_dets = filter_vehicle_detections(
                        result.boxes, class_names, confidence_threshold)  # Keep only vehicle classes
                    detections_for_tracker.extend(filtered_dets)
                bbs = []
                for box_data, cls_id in detections_for_tracker:
                    x1, y1, x2, y2, conf = box_data
                    w, h = x2 - x1, y2 - y1
                    bbs.append([[x1, y1, w, h], conf, cls_id])  # Format for DeepSORT: [bbox, conf, class_id]
                tracks = deepsort.update_tracks(bbs, frame=preprocessed_frame)  # Run tracking

            else:
                tracks = deepsort.update_tracks([], frame=preprocessed_frame)  # Only track, no new detections


            # ---- ROI Counting and Classification Logic ----
            for track in tracks:
                if not track.is_confirmed():
                    continue
                x1, y1, x2, y2 = map(int, track.to_ltrb())   # Get track bounding box
                track_id = track.track_id
                cls_label = track_classes.get(track_id, "Unclassified")  # Get class for this track
                cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)  # Center point of bbox
                mem = track_memory.get(track_id, {"inside_roi": False, "counted": False})
                was_inside = mem["inside_roi"]   # Was this track inside ROI last frame?
                counted = mem["counted"]         # Has this track already been counted?
                now_inside = point_in_roi((cx, cy), roi_polygon)  # Is the center now inside ROI?

                # Only classify and count NOW, at counting time!
                if not was_inside and now_inside and not counted:
                    crop = preprocessed_frame[y1:y2, x1:x2]
                    if crop.shape[0] >= MIN_H and crop.shape[1] >= MIN_W:
                        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                        small = cv2.resize(rgb, (224, 224))
                        arr = img_to_array(small)
                        arr = keras_preprocess(arr)
                        arr = np.expand_dims(arr, axis=0)
                        probs = classifier_model.predict(arr)
                        lab = np.argmax(probs)
                        conf = np.max(probs)
                        if conf >= THRESHOLD:
                            cls_label = custom_classes[lab]
                        else:
                            cls_label = "Unclassified"
                    else:
                        cls_label = "Unclassified"
                    # Store for future drawing
                    track_classes[track_id] = cls_label
                    class_counts[cls_label] += 1
                    print(f"[INFO] Track {track_id} ({cls_label}) entered ROI at frame {frame_idx}")
                    counted = True
                else:
                    # Use label if already classified, else Unclassified
                    cls_label = track_classes.get(track_id, "Unclassified")

                # Update memory for this track for next frame
                track_memory[track_id] = {"inside_roi": now_inside, "counted": counted}
                # # --- Draw bbox and label ---
                # color = (0, 255, 0)
                # cv2.rectangle(preprocessed_frame, (x1, y1), (x2, y2), color, 2)  # Draw bounding box
                # label_text = f"ID:{track_id} {cls_label}"
                # cv2.putText(preprocessed_frame, label_text, (x1, y1 - 10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)  # Draw ID and class

            # # ---- Draw ROI polygon ----
            # cv2.polylines(preprocessed_frame, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)  # Draw ROI area
            # # Show count info at top
            # count_text = " | ".join([f"{k}: {v}" for k, v in class_counts.items()])
            # cv2.putText(preprocessed_frame, count_text, (10, 30),
            #             cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)  # Overlay per-class count
            # # Display & write to output
            # cv2.imshow("YOLO + DeepSORT ROI Counting", preprocessed_frame)
            # # out_writer.write(preprocessed_frame)

        # Keyboard controls
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break  # Quit on 'q'
        elif key == ord('p'):
            paused = not paused
            if paused:
                print("Paused. Press 'c' to continue.")  # Pause
        elif key == ord('c'):
            paused = False
            print("Continuing...")  # Continue after pause

    end_time = time.time()
    elapsed = end_time - start_time
    if elapsed > 0:
        fps = frame_idx / elapsed
        print(f"[INFO] Processed {frame_idx} frames in {elapsed:.2f} seconds")
        print(f"[INFO] Average FPS: {fps:.2f}")

    cap.release()
    # out_writer.release()
    cv2.destroyAllWindows()
    
    totalVehicle = 0
    print("[INFO] --- Final Counts per Class (ROI Crossing) ---")
    for cls, cnt in class_counts.items():
        print(f"{cls}: {cnt}")  # Print final counts for each vehicle class
        totalVehicle += cnt
    print("Total Vehicle Count:", totalVehicle)
    # print(f"[INFO] Processed video saved to: {output_video}")

In [18]:
if __name__ == "__main__":
    main(
        video_source=r"..\cam dataset\Stadium Junctions\Cam1\3-3.15PM\segments_54.mp4",
        resize_dim=(640, 480),
        denoise=False,
        roi_points=[(231, 92), (33, 190), (146, 435), (463, 157), (232, 93)],  # Or e.g. [(x1,y1), (x2,y2), ...] for custom polygon
        output_video="counting_and_tracking.avi",
        detection_interval=1,
        confidence_threshold=0.4
    )

[INFO] ROI Polygon: [(231, 92), (33, 190), (146, 435), (463, 157), (232, 93)]
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[INFO] Track 2 (Motorcycle) entered ROI at frame 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
[INFO] Track 4 (Motorcycle) entered ROI at frame 56
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[INFO] Track 7 (Unclassified) entered ROI at frame 183
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[INFO] Track 8 (Unclassified) entered ROI at frame 311
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
[INFO] Track 6 (High-End Vehicles) entered ROI at frame 505
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
[INFO] Track 11 (Low-End Vehicles) entered ROI at frame 521
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
[INFO] Track 12 (Unclassified) entered ROI at frame 707
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
[INFO] Track 25 (Unclassified) entered ROI at frame 901
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
[INFO] Track 37 (Unclassified) entered ROI at frame 1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
[INFO] Track 39 (Unclassified) entered ROI

In [22]:
if __name__ == "__main__":
    main(
        video_source=r"..\cam dataset\Stadium Junctions\Cam2\3-3.15PM\segments_1.mp4",
        resize_dim=(640, 480),
        denoise=False,
        roi_points=[(246, 161), (627, 343), (589, 476), (6, 229), (247, 161)],  # Or e.g. [(x1,y1), (x2,y2), ...] for custom polygon
        output_video="counting_and_tracking.avi",
        detection_interval=1,
        confidence_threshold=0.4
    )

[INFO] ROI Polygon: [(246, 161), (627, 343), (589, 476), (6, 229), (247, 161)]
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[INFO] Track 1 (Unclassified) entered ROI at frame 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
[INFO] Track 4 (High-End Vehicles) entered ROI at frame 81
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
[INFO] Track 6 (Unclassified) entered ROI at frame 237
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
[INFO] Track 7 (Mid-Range Vehicles) entered ROI at frame 265
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
[INFO] Track 11 (Unclassified) entered ROI at frame 447
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
[INFO] Track 17 (Unclassified) entered ROI at frame 645
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
[INFO] Track 22 (Motorcycle) entered ROI at frame 682
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[INFO] Track 23 (Unclassified) entered ROI at frame 707
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
[INFO] Track 28 (Unclassified) entered ROI at frame 862
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
[INFO] Track 32 (High-End Vehicles) e

# ROI Selection Script

In [21]:
import cv2
import numpy as np

roi_points = []

def mouse_callback(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        roi_points.append((x, y))
        print(f"Point selected: ({x}, {y})")

def select_roi_from_frame(frame):
    temp_frame = frame.copy()
    cv2.namedWindow("Select ROI (click points, press 'q' when done)")
    cv2.setMouseCallback("Select ROI (click points, press 'q' when done)", mouse_callback)

    while True:
        display_frame = preprocess_frame(temp_frame.copy())
        # Draw the ROI as clicking
        for pt in roi_points:
            cv2.circle(display_frame, pt, 5, (0,255,0), -1)
        if len(roi_points) > 1:
            cv2.polylines(display_frame, [np.array(roi_points, np.int32)], isClosed=False, color=(255,0,0), thickness=2)
        cv2.imshow("Select ROI (click points, press 'q' when done)", display_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
    cv2.destroyAllWindows()
    return roi_points

# usage:
video_path = r"..\cam dataset\Stadium Junctions\Cam2\3-3.15PM\segments_1.mp4"
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()
if ret:
    roi_points = select_roi_from_frame(preprocess_frame(frame))
    print("Selected ROI points:", roi_points)
else:
    print("Failed to load video/frame.")


Point selected: (246, 161)
Point selected: (627, 343)
Point selected: (589, 476)
Point selected: (6, 229)
Point selected: (247, 161)
Selected ROI points: [(246, 161), (627, 343), (589, 476), (6, 229), (247, 161)]
